In [ ]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys
import subprocess
from pathlib import Path
import json
import pandas as pd


# Local files/code
import src.data_preprocessing.image_preprocessing as img_pre
import src.data_preprocessing.text_preprocessing as text_pre
import src.data_preprocessing.text_data_exploration as text_explore
import src.data_preprocessing.image_data_exploration as img_explore
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
import src.config as config
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
import src.evaluation.visual_evaluator as visual_evaluator
from src.data_preprocessing.USDA_processing import embed_USDA_data, make_USDA_plant_sheets_dataframe


--------------------- USDA Plant Sheets Text Data Exploration ---------------------------

In [ ]:
# Load USDA text data

usda_plant_sheets= make_USDA_plant_sheets_dataframe(config.Data.USDA.PLANT_SHEETS)

In [ ]:
# explore USDA text data

# When we embed the USDA data, we won't actually preprocess/clean the text very much
# But for the sake of exploring, cleaning and normalization can help us find patterns by removing some noise
text_pre.preprocess_dataframe(
    usda_plant_sheets,
    ["text"],
    [],
    {},
    [],
    "",
)

# explore the cleaned training dataset
text_explore.explore_data(
    usda_plant_sheets, 
    [], 
    ["text"], 
    top_n_words=20, 
    name="USDA Plant Sheets Dataset"
)

--------------------- PlantExpertVQA Data Exploration ---------------------------

In [ ]:
# preprocess (not tokenize) the training, testing, and validation datasets

plant_expert_vqa_TRAIN= text_pre.load_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
plant_expert_vqa_TEST= text_pre.load_csv(config.Data.PlantExpertVQA.TEST_FILE)
plant_expert_vqa_VAL= text_pre.load_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)

# training
text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)


In [ ]:

column_distribution_args= [
    {"column": "crop", "show_counts": False, "figure_size": (10, 5)},
    {"column": "severity", "show_counts": True, "figure_size": (5, 5)},
    {"column": "category", "show_counts": True, "figure_size": (5, 5)},
    {"column": "answer_type", "show_counts": True, "figure_size": (5, 5)},
    {"column": "question_category", "show_counts": False, "figure_size": (10, 5)},
]

# explore the cleaned training dataset
text_explore.explore_data(
    plant_expert_vqa_TRAIN, 
    column_distribution_args, 
    config.Data.PlantExpertVQA.TEXT_COLUMNS, 
    top_n_words=20, 
    name="Plant Expert VQA Training Dataset"
)

In [ ]:
# Most of the images are a small photo pasted onto a large blank canvas, so we crop before resizing
img_explore.show_padding(plant_expert_vqa_TRAIN, config.Data.PlantExpertVQA, count=8)

# The cropped image next to the two channels the mask branch receives
img_explore.show_cropped_and_mask(plant_expert_vqa_TRAIN, config.Data.PlantExpertVQA, count=3)

In [ ]:
traits = ["crop", "disease", "severity"]

classes = {}
for t in traits:
    names = sorted(plant_expert_vqa_TRAIN[t].unique())  # adds names of plants
    classes[t] = names + ["unknown"]  # adds unknown species

train_img = VisualModel.one_row_per_image(plant_expert_vqa_TRAIN, traits)
val_img = VisualModel.one_row_per_image(plant_expert_vqa_VAL, traits)
test_img = VisualModel.one_row_per_image(plant_expert_vqa_TEST, traits)
classes = VisualModel.build_classes(train_img, traits)

train_ds = VisualModel.make_dataset(train_img, classes, config.Data.PlantExpertVQA.ROOT, training=True)
val_ds = VisualModel.make_dataset(val_img, classes, config.Data.PlantExpertVQA.ROOT)

# every image shows up once per question, and those rows do not always agree on a label
img_explore.show_labels(plant_expert_vqa_TRAIN, traits)

------------------ LeafBranch Data Exploration -------------------

In [ ]:
# Load the leafbranch csv files

leafbranch_frames= []

for file in config.Data.LeafBranch.CSV.iterdir():
    if not general_util.is_csv(file):
        Logger.warning(f"Non CSV file found: {file}")
        continue

    frame= text_pre.load_csv(file)
    leafbranch_frames.append(frame)

leafbranch= pd.concat(leafbranch_frames, ignore_index=True)

In [ ]:
# clean the leafbranch data
text_pre.preprocess_dataframe(
    leafbranch,
    config.Data.LeafBranch.TEXT_COLUMNS,
    config.Data.LeafBranch.COLUMNS_TO_REMOVE,
    config.Data.LeafBranch.NA_FILL,
    [],
    config.Data.LeafBranch.ROOT,
)

In [ ]:

column_distribution_args= [
    {"column": "answer", "show_counts": True, "figure_size": (5, 5)},
    {"column": "question_type", "show_counts": True, "figure_size": (5, 5)},
]

# explore the cleaned training dataset
text_explore.explore_data(
    leafbranch, 
    column_distribution_args, 
    config.Data.LeafBranch.TEXT_COLUMNS, 
    top_n_words=20, 
    name="LeafBranch Dataset"
)